In [ ]:
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import xgboost as xgb
from xgboost import callback

# ============================================================
# 1) Chargement JSONL
# ============================================================

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf8") as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

train_df = load_jsonl("../../data/train.jsonl")
test_df = load_jsonl("../../data/kaggle_test.jsonl")

y = train_df["label"]

print("Train:", train_df.shape)
print("Test:", test_df.shape)

# ============================================================

# ------------------------------------------------------------
# 1) IDENTIFIER colonnes qui contiennent dict ou list
# ------------------------------------------------------------
def is_dict_or_list(x):
    return isinstance(x, dict) or isinstance(x, list)

dict_cols_train = [
    c for c in train_df.columns
    if train_df[c].apply(is_dict_or_list).any()
]
dict_cols_test = [
    c for c in test_df.columns
    if test_df[c].apply(is_dict_or_list).any()
]

print("Colonnes à flatten :", dict_cols_train)


# ------------------------------------------------------------
# 2) Fonction robuste pour flatten dict + list
# ------------------------------------------------------------
def robust_expand_column(series, prefix):
    """
    Transforme une colonne contenant dict/list en colonnes scalables.
    """
    expanded_rows = []

    for val in series:
        if isinstance(val, dict):
            expanded_rows.append(val)

        elif isinstance(val, list):
            row = {"len": len(val)}
            for i, item in enumerate(val):
                if isinstance(item, (str, int, float, bool)):
                    row[f"item_{i}"] = item
                else:
                    row[f"item_{i}"] = str(item)
            expanded_rows.append(row)

        else:
            expanded_rows.append({})

    df_expanded = pd.json_normalize(expanded_rows)
    df_expanded = df_expanded.add_prefix(prefix + ".")
    return df_expanded


# ------------------------------------------------------------
# 3) Expansion identique train / test
# ------------------------------------------------------------
def expand_dict_and_list_columns(df, dict_cols):
    df_new = df.copy()
    for col in dict_cols:
        expanded = robust_expand_column(df_new[col], col)
        df_new = pd.concat([df_new.drop(columns=[col]), expanded], axis=1)
    return df_new

train_df = expand_dict_and_list_columns(train_df, dict_cols_train)
test_df  = expand_dict_and_list_columns(test_df , dict_cols_test)


# ------------------------------------------------------------
# 4) REALIGNEMENT colonnes train/test
# ------------------------------------------------------------
train_cols = set(train_df.columns)
test_cols  = set(test_df.columns)

missing = train_cols - test_cols
extra   = test_cols - train_cols

for col in missing:
    test_df[col] = 0

test_df = test_df.drop(columns=list(extra))

train_df = train_df.sort_index(axis=1)
test_df  = test_df.sort_index(axis=1)

print("Colonnes finales :", len(train_df.columns))


# ------------------------------------------------------------
# 5) EXTRACTION FEATURES NUMÉRIQUES + SCALING
# ------------------------------------------------------------
exclude = {"ID", "label", "text"}
num_cols = [c for c in train_df.columns if c not in exclude]

# conversion totale en numérique
train_num = train_df[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
test_num  = test_df[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

scaler = StandardScaler()
X_num_train = scaler.fit_transform(train_num)
X_num_test  = scaler.transform(test_num)

print("Scaling OK — num dims :", X_num_train.shape)


# ------------------------------------------------------------
# 6) TF-IDF + SVD
# ------------------------------------------------------------
text_col = "text"

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=3)
X_tfidf_train = tfidf.fit_transform(train_df[text_col])
X_tfidf_test = tfidf.transform(test_df[text_col])

print("TF-IDF shape :", X_tfidf_train.shape)

svd = TruncatedSVD(n_components=256)
X_tfidf_train = svd.fit_transform(X_tfidf_train)
X_tfidf_test = svd.transform(X_tfidf_test)

print("SVD shape :", X_tfidf_train.shape)


# ------------------------------------------------------------
# 7) CONCAT TF-IDF + FEATURES NUMÉRIQUES
# ------------------------------------------------------------
X_train = np.hstack([X_tfidf_train, X_num_train])
X_test  = np.hstack([X_tfidf_test , X_num_test ])

print("Final dim:", X_train.shape, X_test.shape)

In [ ]:
import xgboost
print(xgboost.__version__)


In [ ]:
# ============================================================
# 1) Train/Validation split
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import xgboost as xgb
import numpy as np

# Split train/validation
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y, test_size=0.1, random_state=42
)

# Fonction de callback pour afficher accuracy tous les 1000 steps
from xgboost.callback import TrainingCallback
from sklearn.metrics import accuracy_score

class AccuracyCallback(TrainingCallback):
    def __init__(self, X_val, y_val, period=1000):
        self.X_val = X_val
        self.y_val = y_val
        self.period = period

    def after_iteration(self, model, epoch, evals_log):
        if (epoch + 1) % self.period == 0:
            y_val_pred = (model.predict(xgb.DMatrix(self.X_val)) >= 0.5).astype(int)
            acc = accuracy_score(self.y_val, y_val_pred)
            print(f"[Step {epoch+1}] Validation accuracy: {acc:.4f}")
        return False  # False pour continuer l'entraînement


# ============================================================
# 2) XGBoost avec XGBClassifier
# ============================================================
xgb_clf = xgb.XGBClassifier(
    objective='binary:logistic',
    n_estimators=2000,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,  # évite les warnings
    eval_metric='logloss',
    random_state=42,
    early_stopping_rounds=50
)

# Entraînement avec early stopping
xgb_clf.fit(
    X_train_split, y_train_split,
    eval_set=[(X_train_split, y_train_split), (X_val, y_val)],
    verbose=50,
)

# ============================================================
# 3) Accuracy finale sur la validation
# ============================================================
y_val_pred = (xgb_clf.predict(X_val) >= 0.5).astype(int)
acc = accuracy_score(y_val, y_val_pred)
print(f"[XGBOOST] Validation accuracy: {acc:.4f}")

# ============================================================
# 4) Prédiction sur le test set
# ============================================================
y_test_pred = (xgb_clf.predict(X_test) >= 0.5).astype(int)

# ============================================================
# 5) Création du fichier submission.csv
# ============================================================

# Vérifier quelles colonnes existent
if "ID" in test_df.columns:
    submission = pd.DataFrame({"ID": test_df["ID"], "Prediction": y_test_pred})
elif "challenge_id" in test_df.columns:
    submission = pd.DataFrame({"ID": test_df["challenge_id"], "Prediction": y_test_pred})
else:
    raise ValueError("Aucune colonne ID valide trouvée dans test_df !")


submission.to_csv("submission_xgboost.csv", index=False)
print("Fichier 'submission_xgboost.csv' généré ✔️")